# M2A · Source simulation and quality

This notebook inspects the simulated silver source tables and the separately generated marketing plan and market context. It does not build the M2B mart or use evaluation-only truth as a model input. Run `uv run --locked mmm-generate-source` from `modeling/` first.

In [ ]:
import json
from datetime import timedelta

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from mmm_lab.paths import DATA_DIR
from mmm_lab.simulation.quality import validate_sources
from mmm_lab.simulation.config import SEGMENT_STATES

manifest = json.loads((DATA_DIR / 'metadata/source_manifest.json').read_text(encoding='utf-8'))
quality = validate_sources(DATA_DIR, save_report=False)
display(pd.DataFrame([{'source': name, **info} for name, info in manifest['files'].items()])[['source', 'rows', 'sha256']])
display(pd.Series(quality, name='source QA'))

## Source record samples

The transaction sample is a detail line, not a basket or a weekly observation. Source identifiers and contact details below are invented.

In [ ]:
for filename, columns in [
    ('source_transactions', ['TRANSACTION_ID', 'STORE_ID', 'BRAND_ID', 'UPC14', 'QUANTITY', 'PRICE', 'TRANSACTION_DATE', 'SUBMISSION_DATE_TIME']),
    ('source_stores', ['SUBMITTER_ID', 'STORE_ID', 'MASTER_STORE_ID', 'STATE', 'STANDARD_STATE']),
    ('source_brands', ['BRAND_ID', 'MASTER_BRAND_ID', 'UPC14', 'DESCRIPTION']),
]:
    path = DATA_DIR / 'silver' / f'{filename}.parquet'
    print(filename, 'rows:', pq.ParquetFile(path).metadata.num_rows)
    display(pq.read_table(path, columns=columns).slice(0, 5).to_pandas())

## Candidate keys and geographic groups

`STANDARD_STATE` determines the six business regions; `STATE` is the documented fallback when standardized geography is missing.

In [ ]:
stores = pq.read_table(DATA_DIR / 'silver/source_stores.parquet', columns=['SUBMITTER_ID', 'STORE_ID', 'MASTER_STORE_ID', 'STATE', 'STANDARD_STATE']).to_pandas()
brands = pq.read_table(DATA_DIR / 'silver/source_brands.parquet', columns=['BRAND_ID', 'UPC14']).to_pandas()
state_to_segment = {state: segment for segment, states in SEGMENT_STATES.items() for state in states}
stores['segment_id'] = stores['STANDARD_STATE'].fillna(stores['STATE']).map(state_to_segment)
display(pd.DataFrame({
    'check': ['unique submitted store key', 'unique submitted product key', 'standard-state fallback', 'missing geographic mapping'],
    'value': [not stores.duplicated(['SUBMITTER_ID', 'STORE_ID']).any(), not brands.duplicated(['BRAND_ID', 'UPC14']).any(), int(stores['STANDARD_STATE'].isna().sum()), int(stores['segment_id'].isna().sum())],
}))
display(stores.groupby('segment_id').size().rename('stores').to_frame())

## Independent event and submission clocks

Event dates govern business-week assignment. Submissions occur later and may fall in a later calendar week.

In [ ]:
timing = pq.read_table(DATA_DIR / 'silver/source_transactions.parquet', columns=['TRANSACTION_DATE', 'SUBMISSION_DATE_TIME', 'MARKET_BASKET_TRANSACTION_ID']).to_pandas()
event = pd.to_datetime(timing['TRANSACTION_DATE'])
submitted = pd.to_datetime(timing['SUBMISSION_DATE_TIME'], utc=True).dt.tz_localize(None)
lag_days = (submitted - event).dt.total_seconds() / 86400
print('Submission lag, days:', lag_days.describe().round(2).to_dict())
print('Lines submitted in a later Monday week:', int(((submitted.dt.normalize() - pd.to_timedelta(submitted.dt.weekday, unit='d')) > (event.dt.normalize() - pd.to_timedelta(event.dt.weekday, unit='d'))).sum()))
print('Baskets with multiple lines:', int((timing.groupby('MARKET_BASKET_TRANSACTION_ID').size() > 1).sum()))

## Planned campaigns and context

These auxiliary tables are synthetic additions, available before the week begins. Planned spend is distinct from discounts redeemed on transaction receipts.

In [ ]:
plans = pq.read_table(DATA_DIR / 'auxiliary/sim_marketing_plan.parquet').to_pandas()
context = pq.read_table(DATA_DIR / 'auxiliary/sim_market_context.parquet').to_pandas()
print('Unique plan keys:', not plans.duplicated(['segment_id', 'week_start', 'channel']).any())
print('Unique context keys:', not context.duplicated(['segment_id', 'week_start']).any())
print('All plans before week:', bool((pd.to_datetime(plans['plan_created_at']).dt.date < plans['week_start']).all()))
print('All context before week:', bool((pd.to_datetime(context['context_as_of']).dt.date < context['week_start']).all()))
display(plans.head(6))
display(context.head(3))
weekly_spend = plans.groupby(['week_start', 'channel'])['planned_spend'].sum().unstack()
ax = weekly_spend.plot(figsize=(10, 4), title='Planned marketing spend across six regions')
ax.set_ylabel('USD per week')
ax.set_xlabel('Week starting Monday')
plt.tight_layout()
plt.show()

## Allocation reconciliation

The true demand and components are stored separately in `ground_truth.json` for evaluation only. The source QA checker verifies that transaction `QUANTITY` sums to each intended segment-week total. There is no model-ready output in M2A.

In [ ]:
assert quality['allocated_source_units'] == quality['total_sales_units']
assert quality['unmatched_store_lines'] == quality['unmatched_product_lines'] == 0
print(f"Reconciled {quality['allocated_source_units']:,} transaction units across {quality['intended_segment_weeks']} segment-weeks.")